In [ ]:
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

In [ ]:
FILE_PATH = "input.txt"

with open(FILE_PATH, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\!\?]", "", text)   # keep only alphanum + sentence-enders
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned = clean_text(raw_text)

In [ ]:
sentence_pattern = re.compile(r"[^.!?]+[.!?]")
sentences = sentence_pattern.findall(cleaned)
sentences = [s.strip() for s in sentences if len(s.split()) >= 4]
sentences = sentences[:500]

print(f"\n[Data]  Using {len(sentences)} sentences")


[Data]  Using 500 sentences


In [ ]:
corpus_text = " ".join(sentences)
words       = corpus_text.split()
print(f"[Data]  Total tokens  : {len(words)}")

[Data]  Total tokens  : 9024


In [ ]:
unique_words = sorted(set(words))
vocab_size   = len(unique_words)
word2idx     = {w: i for i, w in enumerate(unique_words)}
idx2word     = {i: w for w, i in word2idx.items()}
print(f"[Data]  Vocabulary size: {vocab_size}")

[Data]  Vocabulary size: 2158


In [ ]:
SEQ_LEN = 5
X_raw, y_raw = [], []

for i in range(SEQ_LEN, len(words)):
    seq    = words[i - SEQ_LEN : i]
    target = words[i]
    X_raw.append([word2idx[w] for w in seq])
    y_raw.append(word2idx[target])

X_raw = np.array(X_raw)
y_raw = np.array(y_raw)

print(f"[Data]  Total samples  : {len(X_raw)}")

[Data]  Total samples  : 9019


In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw, y_raw, test_size=0.15, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42
)

print(f"[Data]  Train={len(X_train)}  Val={len(X_val)}  Test={len(X_test)}")

[Data]  Train=6316  Val=1350  Test=1353


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, LSTM, Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
y_train_cat = to_categorical(y_train, num_classes=vocab_size)
y_val_cat   = to_categorical(y_val,   num_classes=vocab_size)
y_test_cat  = to_categorical(y_test,  num_classes=vocab_size)

EMBED_DIM  = 64
HIDDEN     = 128
BATCH      = 64
EPOCHS     = 30
DROPOUT    = 0.3

early_stop = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True, verbose=0
)

In [ ]:
rnn_model = Sequential([
    Embedding(vocab_size, EMBED_DIM, input_length=SEQ_LEN),
    SimpleRNN(HIDDEN, return_sequences=False),
    Dropout(DROPOUT),
    Dense(128, activation="relu"),
    Dense(vocab_size, activation="softmax"),
], name="SimpleRNN_Predictor")

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
rnn_model.summary()

Model: "SimpleRNN_Predictor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
rnn_history = rnn_model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.0209 - loss: 7.1588 - val_accuracy: 0.0363 - val_loss: 6.5546
Epoch 2/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.0345 - loss: 6.2332 - val_accuracy: 0.0326 - val_loss: 6.7159
Epoch 3/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.0371 - loss: 5.9434 - val_accuracy: 0.0467 - val_loss: 6.8155
Epoch 4/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.0538 - loss: 5.6512 - val_accuracy: 0.0533 - val_loss: 7.0635
Epoch 5/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.0695 - loss: 5.4143 - val_accuracy: 0.0548 - val_loss: 7.5442
Epoch 6/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.0738 - loss: 5.2245 - val_accuracy: 0.0519 - val_loss: 8.3465


In [ ]:
lstm_model = Sequential([
    Embedding(vocab_size, EMBED_DIM, input_length=SEQ_LEN),
    LSTM(HIDDEN, return_sequences=False),
    Dropout(DROPOUT),
    Dense(128, activation="relu"),
    Dense(vocab_size, activation="softmax"),
], name="LSTM_Predictor")

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
lstm_model.summary()

Model: "LSTM_Predictor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_history = lstm_model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.0257 - loss: 7.2909 - val_accuracy: 0.0348 - val_loss: 6.6034
Epoch 2/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0340 - loss: 6.3156 - val_accuracy: 0.0348 - val_loss: 6.7894
Epoch 3/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0332 - loss: 6.2238 - val_accuracy: 0.0348 - val_loss: 6.9075
Epoch 4/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.0353 - loss: 6.0901 - val_accuracy: 0.0348 - val_loss: 6.9950
Epoch 5/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.0334 - loss: 5.9792 - val_accuracy: 0.0274 - val_loss: 7.1540
Epoch 6/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.0296 - loss: 5.8670 - val_accuracy: 0.0363 - val_loss: 7.3047


In [ ]:
from tensorflow.keras.metrics import SparseTopKCategoricalAccuracy

def evaluate_model(model, name, X, y_cat, y_sparse):
    loss, acc = model.evaluate(X, y_cat, batch_size=BATCH, verbose=0)
    perplexity = np.exp(loss)

    top3_metric = SparseTopKCategoricalAccuracy(k=3)
    preds = model.predict(X, batch_size=BATCH, verbose=0)
    top3_metric.update_state(y_sparse, preds)
    top3_acc = float(top3_metric.result())

    print(f"\n  [{name}] Test Results")
    print(f"  {'─'*40}")
    print(f"  CE Loss        : {loss:.4f}")
    print(f"  Accuracy       : {acc*100:.2f}%")
    print(f"  Perplexity     : {perplexity:.2f}")
    print(f"  Top-3 Accuracy : {top3_acc*100:.2f}%")

    return {"loss": loss, "accuracy": acc, "perplexity": perplexity, "top3": top3_acc}

rnn_scores  = evaluate_model(rnn_model,  "Simple RNN", X_test, y_test_cat, y_test)
lstm_scores = evaluate_model(lstm_model, "LSTM",       X_test, y_test_cat, y_test)


  [Simple RNN] Test Results
  ────────────────────────────────────────
  CE Loss        : 6.4692
  Accuracy       : 3.10%
  Perplexity     : 644.95
  Top-3 Accuracy : 9.02%

  [LSTM] Test Results
  ────────────────────────────────────────
  CE Loss        : 6.5238
  Accuracy       : 3.18%
  Perplexity     : 681.18
  Top-3 Accuracy : 9.02%


In [ ]:
print(f"  {'Metric':<20} {'Simple RNN':>14} {'LSTM':>14}")
print(f"  {'─'*48}")
for metric, label in [
    ("loss",       "CE Loss"),
    ("accuracy",   "Accuracy"),
    ("perplexity", "Perplexity"),
    ("top3",       "Top-3 Accuracy"),
]:
    r = rnn_scores[metric]
    l = lstm_scores[metric]
    if metric in ("accuracy", "top3"):
        print(f"  {label:<20} {r*100:>13.2f}% {l*100:>13.2f}%")
    else:
        print(f"  {label:<20} {r:>14.4f} {l:>14.4f}")

  Metric                   Simple RNN           LSTM
  ────────────────────────────────────────────────
  CE Loss                      6.4692         6.5238
  Accuracy                      3.10%          3.18%
  Perplexity                 644.9476       681.1833
  Top-3 Accuracy                9.02%          9.02%


In [ ]:
def predict_next_word(model, seed_text: str, top_n: int = 3):
    tokens = clean_text(seed_text).split()
    tokens = tokens[-SEQ_LEN:]
    if len(tokens) < SEQ_LEN:
        tokens = [""] * (SEQ_LEN - len(tokens)) + tokens

    indices = []
    for w in tokens:
        if w in word2idx:
            indices.append(word2idx[w])
        else:
            indices.append(0)   # fallback to index 0

    x = np.array([indices])
    preds = model.predict(x, verbose=0)[0]
    top_indices = np.argsort(preds)[::-1][:top_n]
    return [(idx2word[i], round(float(preds[i]), 4)) for i in top_indices]

In [ ]:
demo_seeds = [
    "i miss him so",
    "she picked up the",
    "the phone rang but",
]

for seed in demo_seeds:
    rnn_preds  = predict_next_word(rnn_model,  seed)
    lstm_preds = predict_next_word(lstm_model, seed)
    print(f"\n  Seed : \"{seed}\"")
    print(f"  RNN  : {rnn_preds}")
    print(f"  LSTM : {lstm_preds}")



  Seed : "i miss him so"
  RNN  : [('the', 0.0222), ('and', 0.0213), ('i', 0.0205)]
  LSTM : [('the', 0.0289), ('i', 0.0281), ('and', 0.0252)]

  Seed : "she picked up the"
  RNN  : [('the', 0.0136), ('i', 0.0127), ('and', 0.0127)]
  LSTM : [('the', 0.0251), ('i', 0.0244), ('and', 0.0221)]

  Seed : "the phone rang but"
  RNN  : [('the', 0.0207), ('and', 0.0193), ('i', 0.0192)]
  LSTM : [('the', 0.0275), ('i', 0.0267), ('and', 0.0241)]


In [ ]:
y_train_cat = to_categorical(y_train, num_classes=vocab_size)
y_val_cat   = to_categorical(y_val,   num_classes=vocab_size)
y_test_cat  = to_categorical(y_test,  num_classes=vocab_size)

EMBED_DIM  = 128
HIDDEN     = 256
BATCH      = 64
EPOCHS     = 60
DROPOUT    = 0.3

early_stop = EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True, verbose=0
)

In [ ]:
rnn_model = Sequential([
    Embedding(vocab_size, EMBED_DIM, input_length=SEQ_LEN),
    SimpleRNN(HIDDEN, return_sequences=False),
    Dropout(DROPOUT),
    Dense(128, activation="relu"),
    Dense(vocab_size, activation="softmax"),
], name="SimpleRNN_Predictor")

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
rnn_model.summary()

Model: "SimpleRNN_Predictor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
rnn_history = rnn_model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.0190 - loss: 7.1227 - val_accuracy: 0.0296 - val_loss: 6.5511
Epoch 2/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.0315 - loss: 6.1483 - val_accuracy: 0.0348 - val_loss: 6.8215
Epoch 3/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0479 - loss: 5.8522 - val_accuracy: 0.0452 - val_loss: 7.0214
Epoch 4/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.0585 - loss: 5.5856 - val_accuracy: 0.0570 - val_loss: 7.1962
Epoch 5/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.0672 - loss: 5.3524 - val_accuracy: 0.0563 - val_loss: 7.6411
Epoch 6/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0752 - loss: 5.1734 - val_accuracy: 0.0533 - val_loss: 7.5862
Epoch 7/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0922 - loss: 5.0136 - val_accuracy: 0.0504 - val_loss: 7.8714
Epoch 8/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.1159 - loss: 4.8483 - val_accuracy: 0.0452 - v

In [ ]:
lstm_model = Sequential([
    Embedding(vocab_size, EMBED_DIM, input_length=SEQ_LEN),
    LSTM(HIDDEN, return_sequences=False),
    Dropout(DROPOUT),
    Dense(128, activation="relu"),
    Dense(vocab_size, activation="softmax"),
], name="LSTM_Predictor")

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
lstm_model.summary()

Model: "LSTM_Predictor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_history = lstm_model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.0271 - loss: 7.2249 - val_accuracy: 0.0237 - val_loss: 6.5888
Epoch 2/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step - accuracy: 0.0265 - loss: 6.3001 - val_accuracy: 0.0348 - val_loss: 6.7778
Epoch 3/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - accuracy: 0.0321 - loss: 6.1831 - val_accuracy: 0.0348 - val_loss: 6.8197
Epoch 4/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 63ms/step - accuracy: 0.0305 - loss: 6.0346 - val_accuracy: 0.0267 - val_loss: 6.9148
Epoch 5/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.0359 - loss: 5.8836 - val_accuracy: 0.0393 - val_loss: 7.2700
Epoch 6/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.0488 - loss: 5.7573 - val_accuracy: 0.0415 - val_loss: 7.7228
Epoch 7/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 5s 54ms/step - accuracy: 0.0564 - loss: 5.6043 - val_accuracy: 0.0556 - val_loss: 7.8693
Epoch 8/60
99/99 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - accuracy: 0.0651 - loss: 5.4297 - val_accuracy: 0.0548 - v

**Reasoning**:
Now that both models have been re-trained with the updated hyperparameters, I need to evaluate their performance on the test set using the `evaluate_model` function to get their loss, accuracy, perplexity, and Top-3 accuracy. This will provide the necessary metrics to compare against the baseline and fulfill the subtask.



In [ ]:
rnn_scores_optimized  = evaluate_model(rnn_model,  "Simple RNN Optimized", X_test, y_test_cat, y_test)
lstm_scores_optimized = evaluate_model(lstm_model, "LSTM Optimized",       X_test, y_test_cat, y_test)


  [Simple RNN Optimized] Test Results
  ────────────────────────────────────────
  CE Loss        : 6.4565
  Accuracy       : 2.66%
  Perplexity     : 636.81
  Top-3 Accuracy : 9.02%

  [LSTM Optimized] Test Results
  ────────────────────────────────────────
  CE Loss        : 6.5075
  Accuracy       : 3.18%
  Perplexity     : 670.13
  Top-3 Accuracy : 9.02%


**Reasoning**:
I need to summarize the performance changes by comparing the baseline metrics with the optimized metrics for both models. This can be achieved by printing a formatted table showing the metrics side-by-side for both the original and optimized models.



In [ ]:
print(f"  {'Metric':<20} {'Simple RNN':>14} {'LSTM':>14} {'Optimized RNN':>18} {'Optimized LSTM':>18}")
print(f"  {'─'*70}")
for metric, label in [
    ("loss",       "CE Loss"),
    ("accuracy",   "Accuracy"),
    ("perplexity", "Perplexity"),
    ("top3",       "Top-3 Accuracy"),
]:
    r_orig = rnn_scores[metric]
    l_orig = lstm_scores[metric]
    r_opt  = rnn_scores_optimized[metric]
    l_opt  = lstm_scores_optimized[metric]

    if metric in ("accuracy", "top3"):
        print(f"  {label:<20} {r_orig*100:>13.2f}% {l_orig*100:>13.2f}% {r_opt*100:>17.2f}% {l_opt*100:>17.2f}%")
    else:
        print(f"  {label:<20} {r_orig:>14.4f} {l_orig:>14.4f} {r_opt:>17.4f} {l_opt:>17.4f}")

  Metric                   Simple RNN           LSTM      Optimized RNN     Optimized LSTM
  ──────────────────────────────────────────────────────────────────────
  CE Loss                      6.4692         6.5238            6.4565            6.5075
  Accuracy                      3.10%          3.18%              2.66%              3.18%
  Perplexity                 644.9476       681.1833          636.8085          670.1344
  Top-3 Accuracy                9.02%          9.02%              9.02%              9.02%
